# Preprocessing IndoBERT — Pipeline Minimal
**Input:** `labelled_data_final.csv` (80.163 baris)  
**Output:** `data_preprocessed_indobert.csv`  

## Perbedaan Pipeline IndoBERT vs Seprianto (XGBoost)
| Tahap | Seprianto (XGBoost) | IndoBERT |
|---|---|---|
| P1 Normalisasi newline | ✅ | ✅ |
| P2 Hapus URL/mention/hashtag | ✅ | ✅ |
| P3 Konversi emoji → kata deskriptif | ✅ | ✅ |
| P4 Hapus tanda baca | ✅ | ❌ (dipertahankan) |
| P5 Case folding | ✅ | ❌ (dipertahankan) |
| P6 Tokenisasi | ✅ | ❌ (oleh BertTokenizer) |
| P7 Normalisasi huruf berulang | ✅ | ✅ |
| P8 Koreksi slang | ✅ | ✅ |
| P9 Hapus stopword | ✅ | ❌ (dipertahankan) |
| P10 Stemming | ✅ | ❌ (dipertahankan) |
| P11 Filter KBBI | ✅ | ❌ (dipertahankan) |
| P12 Filter panjang kata | ✅ | ❌ (dipertahankan) |

**Alasan:** IndoBERT sudah dilatih pada teks natural Bahasa Indonesia.
Stemming, stopword removal, dan case folding akan merusak konteks semantik
yang menjadi keunggulan utama model transformer.


## Cell 1 — Setup & Library

In [12]:
import pandas as pd
import numpy as np
import re, os, time, warnings
warnings.filterwarnings('ignore')

from tqdm.notebook import tqdm
tqdm.pandas()

# ── PATH ─────────────────────────────────────────────────────────────────────
BASE_DIR      = r'C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method'
FILE_IN       = os.path.join(BASE_DIR, 'data/processed/labelled_data_filtered.csv')
FILE_OUT      = os.path.join(BASE_DIR, 'data/processed/data_preprocessed_indobert_3.0.csv')

LABEL_MAP   = {'keluhan':0, 'saran':1, 'pujian':2}
INV_MAP     = {v:k for k,v in LABEL_MAP.items()}
CLASS_NAMES = ['keluhan','saran','pujian']

print('Setup selesai!')
print(f'Input : {FILE_IN}')
print(f'Output: {FILE_OUT}')

Setup selesai!
Input : C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\data/processed/labelled_data_filtered.csv
Output: C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\data/processed/data_preprocessed_indobert_3.0.csv


## Cell 2 — Load Data

In [13]:
df = pd.read_csv(FILE_IN)
df = df.dropna(subset=['text','label_pks'])
df['label_enc'] = df['label_pks'].map(LABEL_MAP)
df = df.dropna(subset=['label_enc'])
df['label_enc'] = df['label_enc'].astype(int)

print(f'Total data berlabel: {len(df):,}')
print(f'\nDistribusi label:')
for lbl, cnt in df['label_pks'].value_counts().items():
    pct = cnt/len(df)*100
    bar = chr(9608)*int(pct/3)
    print(f'  {lbl:10s}: {cnt:6,} ({pct:.1f}%) {bar}')

print(f'\nKolom tersedia: {list(df.columns)}')

# Cek confidence score untuk identifikasi manual vs otomatis
if 'confidence' in df.columns:
    df_manual = df[df['confidence']==1.0]
    df_auto   = df[df['confidence']< 1.0]
    print(f'\nData manual  (conf=1.0): {len(df_manual):,}')
    print(f'Data otomatis (conf<1.0): {len(df_auto):,}')

Total data berlabel: 64,349

Distribusi label:
  keluhan   : 41,267 (64.1%) █████████████████████
  pujian    : 14,413 (22.4%) ███████
  saran     :  8,669 (13.5%) ████

Kolom tersedia: ['id', 'timestamp', 'ownerUsername', 'text', 'likesCount', 'postUrl', 'commentUrl', 'source_file', 'label_pks', 'confidence', 'label_enc']

Data manual  (conf=1.0): 18,534
Data otomatis (conf<1.0): 45,815


## Cell 3 — Load Kamus (Slang + Emoji)

In [14]:
# ── Slang dict ────────────────────────────────────────────────────────────────
SLANG_FILE = os.path.join(BASE_DIR, 'dictionaries/slang_dict.csv')

DOMAIN_PROTECT = {
    'bpjs','mbg','djp','coretax','spt','npwp','ktp','nik',
    'login','error','update','upload','download','website',
    'online','offline','server','sistem','aplikasi'
}

if os.path.exists(SLANG_FILE):
    df_slang = pd.read_csv(SLANG_FILE)
    slang_raw = dict(zip(
        df_slang['slang_list'].str.lower().str.strip(),
        df_slang['baku'].str.lower().str.strip()
    ))
    # Filter: jangan ubah domain words, jangan multi-kata
    slang_dict = {
        k:v for k,v in slang_raw.items()
        if k not in DOMAIN_PROTECT and len(v.split())<=3
    }
    print(f'Slang dict: {len(slang_dict):,} kata')
else:
    slang_dict = {}
    print('Slang dict tidak ditemukan — skip')

# ── Emoji dict (kata DESKRIPTIF, bukan nama kelas) ────────────────────────────
# PENTING: pakai kata deskriptif (marah/senang/harap)
# BUKAN nama kelas (keluhan/pujian/saran)
EMOJI_FILE = os.path.join(BASE_DIR, 'dictionaries/emoji_dict.csv')

if os.path.exists(EMOJI_FILE):
    df_em = pd.read_csv(EMOJI_FILE)
    _kls  = {'keluhan':'marah','pujian':'senang','saran':'harap'}
    emoji_dict = {}
    for _, row in df_em.iterrows():
        kata = _kls.get(str(row['klasifikasi']).lower().strip(), '')
        if kata:
            emoji_dict[str(row['emoji_list'])] = kata
    print(f'Emoji dict: {len(emoji_dict):,} emoji → kata deskriptif')
else:
    # Fallback kamus emoji minimal
    emoji_dict = {
        '\U0001F621':'marah', '\U0001F620':'marah', '\U0001F92C':'marah',
        '\U0001F62D':'sedih', '\U0001F614':'sedih', '\U0001F615':'bingung',
        '\U0001F60A':'senang','\U0001F601':'senang','\U0001F64F':'terima kasih',
        '\U0001F44D':'bagus', '\U0001F44F':'bagus', '\U0001F525':'semangat',
        '\U0001F602':'lucu',  '\U0001F923':'lucu',
    }
    print(f'Emoji dict (builtin): {len(emoji_dict):,} emoji')

print('\nSemua kamus siap!')

Slang dict: 14,688 kata
Emoji dict: 144 emoji → kata deskriptif

Semua kamus siap!


## Cell 4 — Definisi Pipeline IndoBERT (5 Tahap)

Pipeline minimal yang mempertahankan konteks semantik untuk IndoBERT:
- **P1** Normalisasi newline
- **P2** Hapus URL, mention, hashtag
- **P3** Konversi emoji → kata deskriptif
- **P4** Normalisasi huruf berulang (`bagusssss` → `baguss`)
- **P5** Koreksi slang (tanpa case folding, tanpa stemming)


In [15]:
def _ws(t):
    """Bersihkan whitespace berlebihan"""
    return re.sub(r' {2,}', ' ', t).strip()

def preprocess_indobert(text):
    """
    Pipeline IndoBERT — 5 Tahap Minimal
    Mempertahankan: huruf kapital, tanda baca, stopword, konteks kalimat
    Tujuan: biarkan BertTokenizer IndoBERT yang menangani tokenisasi
    """
    # P1: Normalisasi newline
    t = re.sub(r'\r\n|\r|\n', ' ', str(text))
    t = _ws(t)

    # P2: Hapus URL, mention, hashtag
    t = re.sub(r'https?://\S+|www\.\S+', '', t)
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'#\w+', '', t)
    t = _ws(t)

    # P3: Konversi emoji → kata deskriptif
    # PENTING: gunakan kata deskriptif (marah/senang/harap)
    # BUKAN nama kelas (keluhan/pujian/saran)
    for em, kata in emoji_dict.items():
        t = t.replace(em, f' {kata} ')
    # Hapus sisa karakter non-ASCII (emoji yang tidak ada di kamus)
    t = re.sub(r'[^\x00-\x7F\u00C0-\u024F]+', ' ', t)
    t = _ws(t)

    # P4: Normalisasi huruf berulang
    # bagusssss → baguss | tidakkk → tidakk
    t = re.sub(r'(.)\1{2,}', r'\1\1', t)
    t = _ws(t)

    # P5: Koreksi slang (pertahankan huruf kapital asli)
    words = t.split()
    result = []
    for w in words:
        wl = w.lower()
        if wl in DOMAIN_PROTECT:
            result.append(w)                    # pertahankan kata domain
        elif wl in slang_dict:
            result.append(slang_dict[wl])       # ganti dengan kata baku
        else:
            result.append(w)                    # pertahankan kata asli
    t = ' '.join(result)
    t = _ws(t)

    return t

# ── Test pipeline ──────────────────────────────────────────────────────────────
tests = [
    'Coretax ERROR mulu gak bisa login udah 3 hari!!! \U0001F621\U0001F621',
    'BPJS nya bagusss bgt makasih udh bantu keluarga saya \U0001F64F',
    'Tolong perbaiki sistem MBG yg sering error di daerah @admin #bpjs',
    'gak bisa daftar akun dr kemarin, tlg segera ditangani pls!!!',
]

print('TEST PIPELINE INDOBERT (5 Tahap)')
print('='*65)
for t in tests:
    hasil = preprocess_indobert(t)
    print(f'Input : {t}')
    print(f'Output: {hasil}')
    print('-'*65)

TEST PIPELINE INDOBERT (5 Tahap)
Input : Coretax ERROR mulu gak bisa login udah 3 hari!!! 😡😡
Output: Coretax ERROR mulu tidak bisa login sudah 3 hari!! marah marah
-----------------------------------------------------------------
Input : BPJS nya bagusss bgt makasih udh bantu keluarga saya 🙏
Output: BPJS nya bagus banget terima kasih sudah bantu keluarga saya senang
-----------------------------------------------------------------
Input : Tolong perbaiki sistem MBG yg sering error di daerah @admin #bpjs
Output: Tolong perbaiki sistem MBG yang sering error di daerah
-----------------------------------------------------------------
Input : gak bisa daftar akun dr kemarin, tlg segera ditangani pls!!!
Output: tidak bisa daftar akun dari kemarin, tolong segera ditangani pls!!
-----------------------------------------------------------------


## Cell 5 — Jalankan Preprocessing
> ⏱ Estimasi waktu: **~10-15 menit** untuk 80.163 baris  
> (Lebih cepat dari pipeline Seprianto karena tidak ada stemming)


In [16]:
print('Menjalankan preprocessing IndoBERT...')
print(f'Total data: {len(df):,} baris')
print(f'Estimasi : ~10-15 menit')
print()

t0 = time.time()
df['text_prep'] = df['text'].progress_apply(preprocess_indobert)
elapsed = (time.time()-t0)/60

print(f'\nPreprocessing selesai dalam {elapsed:.1f} menit!')

# ── Statistik hasil preprocessing ────────────────────────────────────────────
word_counts = df['text_prep'].str.split().str.len()
empty_count = (df['text_prep'].str.strip() == '').sum()
short_count = (word_counts < 2).sum()

print(f'\nStatistik preprocessing:')
print(f'  Teks kosong      : {empty_count:,}')
print(f'  Teks < 2 kata    : {short_count:,}')
print(f'  Rata-rata kata   : {word_counts.mean():.1f}')
print(f'  Median kata      : {word_counts.median():.0f}')
print(f'  Min kata         : {word_counts.min()}')
print(f'  Max kata         : {word_counts.max()}')

# Contoh hasil
print(f'\nContoh 3 hasil preprocessing:')
for _, row in df.sample(3, random_state=42).iterrows():
    print(f'  [{row.label_pks}]')
    print(f'  Asli  : {str(row.text)[:80]}')
    print(f'  Hasil : {str(row.text_prep)[:80]}')
    print()

Menjalankan preprocessing IndoBERT...
Total data: 64,349 baris
Estimasi : ~10-15 menit



  0%|          | 0/64349 [00:00<?, ?it/s]


Preprocessing selesai dalam 0.1 menit!

Statistik preprocessing:
  Teks kosong      : 3,327
  Teks < 2 kata    : 7,208
  Rata-rata kata   : 13.3
  Median kata      : 8
  Min kata         : 0
  Max kata         : 458

Contoh 3 hasil preprocessing:
  [keluhan]
  Asli  : bpjs ketenagakerjaan aja kls aja ribet,apalagi bpjs PBI . ah ribet lah pkknya ri
  Hasil : bpjs ketenagakerjaan saja kelas saja ribet,apalagi bpjs PBI . ah ribet lah pkkny

  [keluhan]
  Asli  : makan tuh yang diluar negeri suruh balik lagi ke indonesia🤣🤣🤣
  Hasil : makan itu yang di luar negeri suruh balik lagi ke indonesia

  [keluhan]
  Asli  : kenapa di daerah saya belum mendapatkan MBG. kendala nya apa, padahal program in
  Hasil : kenapa di daerah saya belum mendapatkan MBG. kendala nya apa, padahal program in



## Cell 6 — Filter & Simpan Output

In [17]:
# ── Filter teks yang terlalu pendek ──────────────────────────────────────────
n_before = len(df)
df_valid  = df[df['text_prep'].str.strip() != ''].copy()
df_valid  = df_valid[df_valid['text_prep'].str.split().str.len() >= 2].copy()
n_after   = len(df_valid)

print(f'Filter hasil:')
print(f'  Sebelum : {n_before:,}')
print(f'  Sesudah : {n_after:,}')
print(f'  Dihapus : {n_before-n_after:,}')

# ── Distribusi label final ────────────────────────────────────────────────────
print(f'\nDistribusi label final:')
for lbl, cnt in df_valid['label_pks'].value_counts().items():
    pct = cnt/len(df_valid)*100
    bar = chr(9608)*int(pct/3)
    print(f'  {lbl:10s}: {cnt:6,} ({pct:.1f}%) {bar}')

# ── Pisahkan berdasarkan confidence ──────────────────────────────────────────
if 'confidence' in df_valid.columns:
    df_manual_out = df_valid[df_valid['confidence']==1.0]
    df_auto_out   = df_valid[df_valid['confidence']< 1.0]
    print(f'\nData manual  (conf=1.0): {len(df_manual_out):,}')
    print(f'Data otomatis (conf<1.0): {len(df_auto_out):,}')

# ── Simpan kolom yang diperlukan ──────────────────────────────────────────────
COLS_KEEP = ['id','ownerUsername', 'postUrl','label_pks','label_enc','confidence']
cols_available = [c for c in COLS_KEEP if c in df_valid.columns]

df_save = df_valid[cols_available + ['text_prep']].copy()
df_save = df_save.rename(columns={'text_prep':'text'})

df_save.to_csv(FILE_OUT, index=False, encoding='utf-8-sig')

print(f'\n✅ File tersimpan: {FILE_OUT}')
print(f'   Total baris   : {len(df_save):,}')
print(f'   Kolom         : {list(df_save.columns)}')

Filter hasil:
  Sebelum : 64,349
  Sesudah : 57,141
  Dihapus : 7,208

Distribusi label final:
  keluhan   : 40,097 (70.2%) ███████████████████████
  pujian    :  8,756 (15.3%) █████
  saran     :  8,288 (14.5%) ████

Data manual  (conf=1.0): 15,477
Data otomatis (conf<1.0): 41,664

✅ File tersimpan: C:\Users\Lenovo\Downloads\skrips_code\xgboost_indobert_method\data/processed/data_preprocessed_indobert_3.0.csv
   Total baris   : 57,141
   Kolom         : ['id', 'ownerUsername', 'postUrl', 'label_pks', 'label_enc', 'confidence', 'text']


## Cell 7 — Verifikasi Output

In [18]:
# Load ulang untuk verifikasi
df_check = pd.read_csv(FILE_OUT)

print('='*55)
print('VERIFIKASI DATA PREPROCESSED INDOBERT')
print('='*55)
print(f'Total baris     : {len(df_check):,}')
print(f'Kolom           : {list(df_check.columns)}')

print(f'\nDistribusi label:')
for lbl, cnt in df_check['label_pks'].value_counts().items():
    pct = cnt/len(df_check)*100
    print(f'  {lbl:10s}: {cnt:6,} ({pct:.1f}%)')

print(f'\nStatistik panjang teks:')
wc = df_check['text'].str.split().str.len()
print(f'  Rata-rata: {wc.mean():.1f} kata')
print(f'  Median   : {wc.median():.0f} kata')

print(f'\n5 contoh teks terpreprocessing:')
for _, row in df_check.sample(5, random_state=42).iterrows():
    print(f'  [{row.label_pks}] {str(row.text)[:90]}')

print(f'\n✅ Verifikasi selesai!')
print(f'File siap digunakan untuk fine-tuning IndoBERT.')

VERIFIKASI DATA PREPROCESSED INDOBERT
Total baris     : 57,141
Kolom           : ['id', 'ownerUsername', 'postUrl', 'label_pks', 'label_enc', 'confidence', 'text']

Distribusi label:
  keluhan   : 40,097 (70.2%)
  pujian    :  8,756 (15.3%)
  saran     :  8,288 (14.5%)

Statistik panjang teks:
  Rata-rata: 14.9 kata
  Median   : 9 kata

5 contoh teks terpreprocessing:
  [keluhan] mengerti kan kenapa tiba dimatiin fasilitasnya wkwk
  [keluhan] Hapus peraturan permenkes No. tahun ! Kami kerja bulan dan sebelum kami terima gaji sudah 
  [keluhan] Di stop saja pak boross lah ini
  [keluhan] saudara aku jadi supplier mbg, ada beberapa dapur yang cheffnya pesan kg tapi meminta ditu
  [keluhan] omongono mas pak pur, mbalek manual ae.. aku ra dong sama sekali ..

✅ Verifikasi selesai!
File siap digunakan untuk fine-tuning IndoBERT.
